# Week 1 — Data Exploration
**Project:** spatial-avm — Geospatial Valuation via Spatial Embeddings
**Author:** Vaibhav Gautam (Baseline ML Engineer)
**Date:** Week 1, Day 1 (Monday)
**Issue:** #2

## Objective
Review the King County Housing dataset's schema, null structure, and dtypes
to plan the data-cleaning approach for Day 3 (Wednesday).

## 1. Environment Setup & Imports

In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
import os

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 150)

print("pandas:", pd.__version__)
print("geopandas:", gpd.__version__)
print("numpy:", np.__version__)

## 2. Load Dataset

Dataset: King County House Sales (Kaggle: harlfoxem/housesalesprediction)
Path is relative to `notebooks/`, and `data/` is gitignored per project rules —
this file is never committed to the repo.

In [ ]:
DATA_PATH = "../data/kc_house_data.csv"

if os.path.exists(DATA_PATH):
    df = pd.read_csv(DATA_PATH)
    print(f"Loaded {df.shape[0]} rows and {df.shape[1]} columns.")
else:
    df = None
    print(f"⚠️ Dataset not found at {DATA_PATH}. Download it before running full EDA.")

## 3. First Look at the Data

In [ ]:
if df is not None:
    display(df.head())

In [ ]:
if df is not None:
    display(df.sample(5, random_state=42))

## 4. Schema Overview — Column Names, Types, Non-Null Counts

In [ ]:
if df is not None:
    df.info()

In [ ]:
if df is not None:
    display(pd.DataFrame({
        "dtype": df.dtypes,
        "n_unique": df.nunique(),
        "n_missing": df.isnull().sum()
    }))

## 5. Missing Value Audit

In [ ]:
if df is not None:
    missing = df.isnull().sum()
    missing_pct = (missing / len(df)) * 100
    missing_summary = pd.DataFrame({
        "missing_count": missing,
        "missing_pct": missing_pct
    }).sort_values("missing_count", ascending=False)
    display(missing_summary[missing_summary["missing_count"] > 0])
    if missing_summary["missing_count"].sum() == 0:
        print("✅ No missing values found in any column.")

## 6. Descriptive Statistics — Numerical Columns

In [ ]:
if df is not None:
    display(df.describe().T)

## 7. Target Variable — Price Distribution

In [ ]:
if df is not None and "price" in df.columns:
    print("Price summary:")
    print(df["price"].describe())
    print("\nSkewness:", df["price"].skew())
    print("Kurtosis:", df["price"].kurt())

## 8. Geographic Columns Check (lat/long)

These are critical — Day 4's haversine distance function and Week 3's
KNN graph construction both depend on clean, valid coordinates.

In [ ]:
if df is not None:
    geo_cols = [c for c in df.columns if c.lower() in ("lat", "long", "lng", "longitude", "latitude")]
    print("Detected geo columns:", geo_cols)
    if geo_cols:
        display(df[geo_cols].describe())

In [ ]:
# King County approximate bounding box sanity check
if df is not None and "lat" in df.columns and "long" in df.columns:
    lat_bounds = (47.1, 47.8)
    long_bounds = (-122.6, -121.3)

    out_of_bounds = df[
        (~df["lat"].between(*lat_bounds)) | (~df["long"].between(*long_bounds))
    ]
    print(f"Rows outside expected King County bounds: {len(out_of_bounds)}")
    display(out_of_bounds.head())

## 9. Duplicate Records Check

In [ ]:
if df is not None:
    dupes = df.duplicated().sum()
    print(f"Fully duplicated rows: {dupes}")
    if "id" in df.columns:
        id_dupes = df["id"].duplicated().sum()
        print(f"Duplicate property IDs: {id_dupes}")

## 10. Candidate Outlier Columns (for Day 2 profiling)

In [ ]:
if df is not None:
    candidate_cols = [c for c in ["price", "sqft_living", "sqft_lot", "bedrooms", "bathrooms"] if c in df.columns]
    for col in candidate_cols:
        q1, q3 = df[col].quantile([0.25, 0.75])
        iqr = q3 - q1
        lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        n_outliers = ((df[col] < lower) | (df[col] > upper)).sum()
        print(f"{col}: IQR bounds [{lower:.1f}, {upper:.1f}] → {n_outliers} potential outliers")

## 11. Initial Observations

*(Fill in after running against the real dataset)*

- **Row / column count:** ...
- **Missing values:** ...
- **Lat/long columns:** present as `lat`, `long`; range looks valid / has outliers (TBD)
- **Price distribution:** right-skewed as expected for housing data (TBD confirm)
- **Duplicate records:** ...
- **Columns likely needing cleaning:** ...
- **Columns likely useful for baseline features (Week 2):** sqft_living, bedrooms, bathrooms, yr_built, yr_renovated, distance-to-center (from Day 4)

## Next Steps
- Day 2 (Tue): Formal outlier profiling using IQR/z-score → defines Day 3 cleaning rules
- Day 3 (Wed): Apply cleaning based on this schema + Day 2 outlier flags
- Day 4 (Thu): Implement `haversine_distance()` in `src/geo_utils.py`

## 12. Outlier Profiling — Price & Square Footage (Day 2)
Using IQR and Z-score methods to flag extreme values.
This defines the cleaning rules Member 1 (Tarun) applies on Day 3.

In [ ]:
from scipy import stats

target_cols = ["price", "sqft_living"]
outlier_report = {}

In [ ]:
def iqr_outliers(series, k=1.5):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - k * iqr, q3 + k * iqr
    mask = (series < lower) | (series > upper)
    return mask, lower, upper

for col in target_cols:
    mask, lower, upper = iqr_outliers(df[col])
    outlier_report[col] = {
        "method": "IQR",
        "lower_bound": lower,
        "upper_bound": upper,
        "n_outliers": mask.sum(),
        "pct_outliers": round(mask.sum() / len(df) * 100, 2)
    }

pd.DataFrame(outlier_report).T

In [ ]:
def zscore_outliers(series, threshold=3):
    z = np.abs(stats.zscore(series, nan_policy="omit"))
    mask = z > threshold
    return mask

for col in target_cols:
    mask = zscore_outliers(df[col])
    outlier_report[col]["zscore_n_outliers"] = mask.sum()
    outlier_report[col]["zscore_pct_outliers"] = round(mask.sum() / len(df) * 100, 2)

pd.DataFrame(outlier_report).T

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, col in zip(axes, target_cols):
    ax.boxplot(df[col].dropna())
    ax.set_title(f"Boxplot: {col}")
plt.tight_layout()
plt.show()